# 22 — Agent SecOps, Detection & Incident Response

## Learning requirements
Production security không kết thúc khi deploy. Bạn phải biết:
- audit event nào cần lưu;
- anomaly nào phải alert;
- cách stop/disable agent capability;
- credential revocation;
- memory quarantine/purge;
- checkpoint rollback/time travel;
- incident triage, containment, recovery và postmortem;
- cách phân biệt model-quality incident, tool incident, security incident và provider outage.

## Operational principle
Một autonomous agent có quyền side effect cần **kill switch + capability kill switch**, không chỉ một nút stop UI.

## Audit trail

Tối thiểu log structured metadata cho:
- request/user/tenant/thread/run IDs;
- model/provider/version;
- prompt/policy/tool/skill versions;
- retrieved source IDs;
- tool proposed/executed;
- target resource;
- policy allow/deny/approval;
- approver + decision;
- memory writes/deletes;
- sandbox/session ID;
- latency/tokens/cost;
- exception/retry/fallback;
- security alert classification.

Không log raw secret. PII/sensitive traces phải tuân theo data policy từ notebook 19.

In [ ]:
from dataclasses import dataclass
from enum import Enum

class RunMode(str, Enum):
    NORMAL = "normal"
    READ_ONLY = "read_only"
    DISABLED = "disabled"

@dataclass
class SafetyControls:
    mode: RunMode = RunMode.NORMAL
    disabled_tools: set[str] = None

    def __post_init__(self):
        if self.disabled_tools is None:
            self.disabled_tools = set()

    def tool_allowed(self, tool: str, is_write: bool) -> bool:
        if self.mode == RunMode.DISABLED or tool in self.disabled_tools:
            return False
        if self.mode == RunMode.READ_ONLY and is_write:
            return False
        return True

controls = SafetyControls()
assert controls.tool_allowed("search_docs", False)
controls.disabled_tools.add("publish_spec")
assert not controls.tool_allowed("publish_spec", True)
controls.mode = RunMode.READ_ONLY
assert not controls.tool_allowed("save_memory", True)

## Detection ideas

Alert candidates:
- unexpected privileged tool sequence;
- repeated deny decisions;
- sudden tool/model call spike;
- unusual cross-tenant resource attempts;
- memory-write burst;
- new/unknown MCP server;
- sandbox outbound host not in baseline;
- repeated HITL requests for destructive actions;
- cost/latency anomaly;
- security-eval or online-eval regression.

Không dùng LLM-only detector cho critical controls; combine deterministic signals với semantic analysis khi cần.

In [ ]:
def detect_anomalies(run: dict) -> list[str]:
    alerts = []
    if run.get("denied_tool_calls", 0) >= 3:
        alerts.append("repeated_policy_denials")
    if run.get("model_calls", 0) > 25:
        alerts.append("model_call_spike")
    if run.get("cross_tenant_attempts", 0) > 0:
        alerts.append("cross_tenant_attempt")
    if run.get("unknown_mcp_server", False):
        alerts.append("unknown_mcp_server")
    return alerts

sample = {"denied_tool_calls": 4, "model_calls": 8, "cross_tenant_attempts": 1}
print(detect_anomalies(sample))

## Incident-response sequence

```text
Detect
  -> Triage severity/scope
  -> Contain: disable run/tool/write/network
  -> Revoke/rotate credentials if exposed
  -> Preserve evidence/traces/checkpoints
  -> Quarantine poisoned memory/index/artifacts
  -> Eradicate root cause
  -> Recover from known-good config/checkpoint
  -> Run functional + security regression suites
  -> Gradual re-enable / canary
  -> Postmortem + new regression case
```

Mọi real incident nên tạo ít nhất một permanent regression test.

## Exercise — Tabletop incident

Giả lập scenario an toàn:
1. một repository fixture chứa indirect injection;
2. agent liên tục đề xuất privileged tool nhưng policy chặn;
3. detector phát hiện repeated denials;
4. operator chuyển hệ thống sang READ_ONLY;
5. quarantine source/memory liên quan;
6. rerun security regression;
7. re-enable after pass.

## Required output
- `artifacts/security/incident-response-runbook.md`
- `artifacts/security/security-alert-catalog.md`
- kill-switch/capability-disable design;
- one tabletop report.

## Done criteria
- Có cách disable toàn agent và từng privileged capability.
- Có credential-revocation path.
- Có memory/vector-store quarantine/cleanup path.
- Recovery bắt buộc chạy security regression trước khi full enable.
- Audit evidence đủ để reconstruct who/what/when/resource/policy decision.